# Part 17 — Legged Locomotion and Humanoids

Legged locomotion combines dynamics, contacts, control, state estimation, simulation, and RL. It is one of the hardest embodied-AI domains.

**Learning style:** mechanisms first → frameworks second → real systems third. The notebook is intentionally slow, explicit, and beginner-friendly.

In [ ]:
# Setup: run this first.
# Works from the repository root. In Colab, clone the repo first, then run from inside it.
from pathlib import Path
import sys, math, random
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    print('Tip: run this notebook from the repository root, or clone the repo in Colab first.')
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)

## Notebook type and ordered study structure

**Notebook type:** Type C — real-system/application notebook. This should help you combine many components into robotics, autonomous-driving, drone, humanoid, or VLA systems.

**Your objective in this notebook:** Combine dynamics, contact, control, and RL for walking robots.

Use this exact order every time:

1. **Why this matters** — identify the real problem this topic solves.
2. **Mental model** — explain the idea in plain language before symbols.
3. **Math mechanism** — write the smallest formula and define every symbol.
4. **From-scratch code** — run/read the simple implementation slowly.
5. **Line-by-line explanation** — trace inputs, internal variables, update rule, and outputs.
6. **Debug/visualize** — print shapes, values, curves, maps, or weights so behavior is visible.
7. **Framework version** — map the mechanism to a real library/API.
8. **Scratch → framework mapping** — write what the framework hides and what it exposes.
9. **Real-system role** — place the topic inside robotics, driving, drones, manipulation, or VLA.
10. **Failure modes** — list how it breaks and how you would notice.
11. **Exercises** — change parameters, break the example, and explain the result.
12. **Mini-project** — build a small artifact you can keep.
13. **Next step** — choose the next notebook or tool.

**Mental model for this topic:** Walking is controlled falling with periodic structure and contact switching.

**Core math / mechanism to keep in mind:** phase_next=(phase+omega dt) mod 2pi.

**Recommended debug habit:** after every code cell, ask “what are the inputs, what changed, and what would be unsafe or wrong in a real robot?”

## From-scratch focus and code-reading checklist

**Scratch focus:** Alternating leg phase toy example.

When reading code in this notebook or the matching repo module, trace it like this:

| Step | Question to answer |
|---|---|
| Input | What is the state, observation, tensor, point, reward, or measurement? |
| Representation | Is it a scalar, vector, matrix, image, point cloud, token sequence, or action? |
| Mechanism | Which line implements the math/update rule? |
| Parameters | Which numbers are hyperparameters, physical constants, or learned weights? |
| Output | What changed after the step? |
| Debug signal | What should I print/plot to know it is working? |

Do not move to the framework/API version until you can explain the scratch version without reading the code comments.

## Visualization and debugging ideas

Use at least one of these while studying:

- Print tensor/vector shapes before and after the core operation.
- Print the first few values before and after an update.
- Plot a curve when there is learning, control, filtering, planning, or optimization.
- Draw frames, maps, paths, sensor rays, or attention matrices when geometry is involved.
- Change one parameter at a time and predict the effect before running.

For this notebook, a useful first visualization/debug target is: **Alternating leg phase toy example.**

## Scratch → framework mapping

| From-scratch idea in this repo | Practical framework/API | What to learn from the framework |
|---|---|---|
| phase toy | gait generators | controllers often use phase structure |
| RL policy | Isaac Lab legged tasks | massive parallel simulation trains locomotion |
| toy dynamics | MuJoCo/robot SDKs | hardware needs safety and state estimation |

**Framework learning rule:** do not memorize the API first. First identify which scratch concept it replaces, then learn its inputs, outputs, configuration, and failure modes.

## Real-system application

Legged locomotion is a synthesis of simulation, control, estimation, and RL.

Ask these system questions:

1. What module produces the input to this component?
2. What module consumes its output?
3. What latency, safety, calibration, or data-format assumptions exist?
4. What metric tells me this component is good enough for the larger system?

## Failure modes and debugging

Common ways this topic can fail:

- contact sim errors
- falls during exploration
- state estimator drift
- actuator overheating

For each failure, write:

- **Symptom:** what would I see in logs, plots, robot behavior, or evaluation?
- **Likely cause:** what assumption broke?
- **First debug action:** what is the smallest thing to inspect?

## Mini-project and mastery checklist

**Mini-project:** Design observations/actions/rewards for a quadruped stand task.

Mastery checklist:

- [ ] I can explain the mental model in one paragraph.
- [ ] I can write the core formula and define every symbol.
- [ ] I can run or read the scratch code and point to the core update/operation.
- [ ] I can name the production framework/API version of the same idea.
- [ ] I can describe where this topic sits in a robot/car/drone/VLA stack.
- [ ] I can name at least three failure modes and one debug action for each.

**Next study steps:** Part 2 physics, Part 14 control, RL 15 safe sim-to-real

## 1. Mental model

Legged locomotion combines dynamics, contacts, control, state estimation, simulation, and RL. It is one of the hardest embodied-AI domains.

Before code, write one sentence in your own words: *what problem does this topic solve?*

## 2. Mechanism and math

Walking is controlled falling. A simple gait can be represented by periodic phase:
\[
phase_{t+1}=(phase_t+\omega dt) \bmod 2\pi
\]
Policies often learn residuals over rhythmic or controller-based structure.

## 3. From-scratch lab

Generate simple alternating leg phases.

Read every line. The code avoids clever abstractions so you can see the mechanism.

In [ ]:
import math
phase = 0.0
dt = 0.1
omega = 2*math.pi
for step in range(12):
    left = math.sin(phase)
    right = math.sin(phase + math.pi)
    print(f'{step:02d} left={left:+.2f} right={right:+.2f}')
    phase = (phase + omega*dt) % (2*math.pi)

## 3.1 Code reading guide

When you read the previous cell, do not treat it as a black box. Trace it in this order:

1. **Inputs:** what are the given numbers, observations, states, rewards, or measurements?
2. **Internal variables:** what does each variable represent physically or mathematically?
3. **Update rule:** which line is the core mechanism from the math section?
4. **Output:** what should change if the mechanism is working?
5. **Failure case:** what parameter could make the example unstable, wrong, or unsafe?

This habit is the bridge between toy examples and real robotics code: every simulator, ROS node, policy, controller, or perception model still has inputs, state, an update rule, and outputs.

## 4. Framework/practice view

Frameworks: Isaac Lab for massive parallel legged RL, MuJoCo for physics experiments, Unitree/robot SDKs for hardware, ROS 2 for integration.

The goal is not to replace understanding with APIs. The goal is to recognize the same mechanism when a library hides the details.

In [ ]:
print('Framework practice: study Isaac Lab locomotion task configs: observations, actions, rewards, domain randomization.')

## 4.1 Framework comparison checklist

After running or reading the framework cell, write a small mapping table for yourself:

| Question | Your answer |
|---|---|
| What object/function in the framework replaces the scratch code? |  |
| Which parameters match the math symbols? |  |
| What details does the framework hide? |  |
| What new engineering concerns appear? | installation, devices, logging, data formats, batching, safety, versioning |

This is where top-down learning becomes useful: you learn the professional API **without losing the mechanism**

## 5. Real-system connection

Humanoid systems need low-level stabilization, contact handling, state estimation, policy training, safety checks, and careful sim-to-real transfer.

## 6. Exercises

1. What observations does a quadruped policy need?
2. Why is contact hard to simulate?
3. What safety tests would you require before hardware deployment?

**Notebook habit:** after each exercise, add a short note explaining what changed and why it matters in a robot/car/drone/VLA stack.